# 人工プロモーター設計ツール - 詳細説明版

このノートブックでは、プロモーター設計の仕組みを図と詳しい説明で学べます。

# コードの詳しい説明（Python初学者向け）

## 📚 このコードが何をしているか

このプログラムは、**人工プロモーター**（遺伝子の発現を制御するDNA配列）を大量に設計するツールです。

---

## 🔍 主要な関数の説明

### 1. `Coplanar()` と `Opposite()` 関数

```python
def Coplanar(core_promoter, TFB, spacer_length=10):
    if not core_promoter or not TFB:
        raise ValueError("core_promoterとTFBは空であってはいけません")
    spacer = 'N' * spacer_length
    artificial_promoter = TFB + spacer + core_promoter
    return artificial_promoter
```

**何をしている？**
- TFB（転写因子結合部位）とコアプロモーターをつなげています
- 間にスペーサー（'N'を繰り返した配列）を挟みます

**Python文法のポイント：**
- `def` で関数を定義します
- `spacer_length=10` はデフォルト引数（指定しなければ10になる）
- `'N' * 10` は文字列を繰り返す（`'NNNNNNNNNN'`になる）
- `TFB + spacer + core_promoter` は文字列を連結する
- `return` で結果を返す

---

### 2. `generate_promoter_library()` 関数 ⭐️ 重要！

この関数が核心部分です。大量のプロモーターを自動生成します。

#### ステップ1: データの読み込み

```python
if core_csv:
    core_df = pd.read_csv(core_csv)
    core_col = 'sequence' if 'sequence' in core_df.columns else core_df.columns[0]
    cores = core_df[core_col].tolist()
elif core_list:
    cores = core_list
```

**何をしている？**
- CSVファイルがあれば、pandasで読み込む
- `'sequence'` というカラム名があればそれを使い、なければ最初のカラムを使う
- `.tolist()` でDataFrameのカラムをリストに変換

**Python文法のポイント：**
- `if ... elif ... else` は条件分岐
- `pd.read_csv()` はpandasでCSVを読み込む関数
- `'sequence' in core_df.columns` は「sequenceがカラム名に含まれているか」チェック
- `.tolist()` はDataFrameの列をPythonのリストに変換

---

#### ステップ2: 全組み合わせの生成 🔄

```python
from itertools import product

for i, (core, tfb) in enumerate(product(cores, tfbs), 1):
```

**何をしている？**
- `product(cores, tfbs)` は全組み合わせを作る
  - 例: cores=['A', 'B']、tfbs=['X', 'Y'] → [('A','X'), ('A','Y'), ('B','X'), ('B','Y')]
- `enumerate(..., 1)` は番号を1から付ける

**Python文法のポイント：**
- `itertools.product()` は直積（すべての組み合わせ）を生成
- `enumerate()` はループしながら番号を付ける
- `for i, (core, tfb) in ...` は複数の値を同時に取り出す（アンパック）

**具体例：**
```python
cores = ['TATAAA', 'TATAAT']
tfbs = ['CAAT', 'GCCAAT']

# product()は以下を生成:
# ('TATAAA', 'CAAT')
# ('TATAAA', 'GCCAAT')
# ('TATAAT', 'CAAT')
# ('TATAAT', 'GCCAAT')
```

---

#### ステップ3: 各組み合わせでプロモーターを設計

```python
entry = {
    'ID': f'Promoter_{i:05d}',
    'Core_Promoter': core,
    'TFB': tfb,
}

if 'Coplanar' in promoter_types:
    coplanar_seq = Coplanar(core, tfb, spacer_length=coplanar_spacer)
    entry['Coplanar_Sequence'] = coplanar_seq
    entry['Coplanar_Length'] = len(coplanar_seq)
```

**何をしている？**
- 辞書（dictionary）に各プロモーターの情報を格納
- Coplanar関数を呼び出して配列を生成
- 配列とその長さを辞書に追加

**Python文法のポイント：**
- `{}` は辞書（キーと値のペア）
- `f'Promoter_{i:05d}'` はf-string（フォーマット済み文字列）
  - `i:05d` は「5桁の数字、0で埋める」（例: 00001, 00042）
- `entry['キー'] = 値` で辞書に追加
- `len()` は文字列の長さを取得

---

#### ステップ4: DataFrameに変換して保存

```python
library = []
for ...:
    entry = {...}
    library.append(entry)

library_df = pd.DataFrame(library)

if output_csv:
    library_df.to_csv(output_csv, index=False)
```

**何をしている？**
- 空のリスト `library = []` を作成
- ループで各プロモーター情報（辞書）を追加
- 最後に全部をDataFrameに変換
- CSVファイルとして保存

**Python文法のポイント：**
- `list.append()` はリストに要素を追加
- `pd.DataFrame(リスト)` は辞書のリストをDataFrameに変換
- `.to_csv()` はDataFrameをCSVファイルに保存
- `index=False` は行番号を保存しない

---

## 🎯 実際の処理の流れ（例）

仮に以下のデータがあるとします：
```python
cores = ['TATAAA', 'TATAAT']  # 2種類
tfbs = ['CAAT', 'GCCAAT']     # 2種類
```

### 処理の流れ：

1. **組み合わせ生成**: 2 × 2 = **4組**
   ```
   (TATAAA, CAAT)
   (TATAAA, GCCAAT)
   (TATAAT, CAAT)
   (TATAAT, GCCAAT)
   ```

2. **各組でプロモーターを設計**:
   - 組1: `CAAT + NNNNNNNNNN + TATAAA` (Coplanar)
   - 組1: `CAAT + NNNNN + TATAAA` (Opposite)
   - 組2: `GCCAAT + NNNNNNNNNN + TATAAA` (Coplanar)
   - ... 以下同様

3. **DataFrameに格納**:
   ```
   | ID          | Core_Promoter | TFB    | Coplanar_Sequence      | Opposite_Sequence  |
   |-------------|---------------|--------|------------------------|---------------------|
   | Promoter_00001 | TATAAA     | CAAT   | CAATNNNNNNNNNNNTATAAA  | CAATNNNNNTATAAA    |
   | Promoter_00002 | TATAAA     | GCCAAT | GCCAATNNNNNNNNNNNTATAAA| GCCAATNNNNNTATАAA  |
   | ...         | ...           | ...    | ...                    | ...                 |
   ```

4. **CSVファイルに保存**

---

## 💡 重要なPython概念のまとめ

| 概念 | 説明 | 例 |
|------|------|-----|
| **f-string** | 変数を文字列に埋め込む | `f'Hello {name}'` |
| **辞書** | キーと値のペア | `{'name': 'Alice', 'age': 25}` |
| **リスト** | 順序付きのデータ集合 | `[1, 2, 3, 4]` |
| **DataFrame** | 表形式のデータ（pandas） | Excelのような表 |
| **product()** | 全組み合わせを生成 | `product([1,2], ['a','b'])` → `[(1,'a'), (1,'b'), (2,'a'), (2,'b')]` |
| **enumerate()** | ループに番号を付ける | `enumerate(['a','b'])` → `[(0,'a'), (1,'b')]` |

---

## 🚀 実際の使い方

### パターン1: リストから直接指定
```python
library = generate_promoter_library(
    core_list=['TATAAA', 'TATAAT'],
    tfb_list=['CAAT', 'GCCAAT'],
    output_csv='my_library.csv'
)
```

### パターン2: CSVファイルから読み込み
```python
# core_promoters.csv の例:
# sequence
# TATAAA
# TATAAT
# TTGACA

library = generate_promoter_library(
    core_csv='core_promoters.csv',
    tfb_csv='tfb_sites.csv',
    output_csv='promoter_library.csv'
)
```

---

## 📊 スケール感

- **100種類のCore** × **50種類のTFB** = **5,000個のプロモーター**
- **1,000種類のCore** × **100種類のTFB** = **100,000個のプロモーター**

このコードなら、数千〜数万のプロモーターを数秒で生成できます！

In [ ]:
# Create conceptual diagram for promoter design (English version)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# ===== Figure 1: Basic Concept of Promoter Design =====
ax1 = axes[0]
ax1.set_xlim(0, 15)
ax1.set_ylim(0, 8)
ax1.axis('off')
ax1.set_title('Basic Concept of Promoter Design', fontsize=16, fontweight='bold', pad=20)

# Coplanar type explanation
y_coplanar = 5.5
# TFB
tfb_box = FancyBboxPatch((1, y_coplanar), 2, 0.8, boxstyle="round,pad=0.05", 
                          edgecolor='blue', facecolor='lightblue', linewidth=2)
ax1.add_patch(tfb_box)
ax1.text(2, y_coplanar+0.4, 'TFB', ha='center', va='center', fontsize=12, fontweight='bold')
ax1.text(2, y_coplanar-0.5, '(TF Binding Site)', ha='center', va='center', fontsize=9)

# Spacer
spacer_box = FancyBboxPatch((3.2, y_coplanar), 2, 0.8, boxstyle="round,pad=0.05",
                             edgecolor='gray', facecolor='lightgray', linewidth=2)
ax1.add_patch(spacer_box)
ax1.text(4.2, y_coplanar+0.4, 'Spacer', ha='center', va='center', fontsize=12, fontweight='bold')
ax1.text(4.2, y_coplanar-0.5, '(10 or 5 bases)', ha='center', va='center', fontsize=9)

# Core Promoter
core_box = FancyBboxPatch((5.4, y_coplanar), 2.5, 0.8, boxstyle="round,pad=0.05",
                           edgecolor='red', facecolor='lightcoral', linewidth=2)
ax1.add_patch(core_box)
ax1.text(6.65, y_coplanar+0.4, 'Core Promoter', ha='center', va='center', fontsize=12, fontweight='bold')
ax1.text(6.65, y_coplanar-0.5, '(Core Promoter)', ha='center', va='center', fontsize=9)

# Labels
ax1.text(0.3, y_coplanar+0.4, "5'", ha='center', va='center', fontsize=14, fontweight='bold')
ax1.text(8.3, y_coplanar+0.4, "3'", ha='center', va='center', fontsize=14, fontweight='bold')
ax1.text(4.4, y_coplanar+1.5, 'Coplanar / Opposite Type', ha='center', va='center', 
         fontsize=13, fontweight='bold', style='italic')

# Example
y_example = 3.5
ax1.text(0.5, y_example+0.8, '[Example]', ha='left', va='center', fontsize=12, fontweight='bold')

# Input
ax1.text(1, y_example+0.3, 'TFB:', ha='left', va='center', fontsize=11, fontfamily='monospace')
ax1.text(2.5, y_example+0.3, 'GCCAAT', ha='left', va='center', fontsize=11, 
         fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

ax1.text(1, y_example-0.3, 'Core:', ha='left', va='center', fontsize=11, fontfamily='monospace')
ax1.text(2.5, y_example-0.3, 'TATAAA', ha='left', va='center', fontsize=11,
         fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

# Arrow
arrow1 = FancyArrowPatch((4.5, y_example), (6, y_example), 
                         arrowstyle='->', mutation_scale=20, linewidth=2, color='green')
ax1.add_patch(arrow1)

# Output
ax1.text(6.5, y_example+0.3, 'Coplanar (10 bases):', ha='left', va='center', fontsize=10, fontfamily='monospace')
ax1.text(6.5, y_example-0.1, 'GCCAAT', ha='left', va='center', fontsize=9,
         fontfamily='monospace', color='blue', fontweight='bold')
ax1.text(8.3, y_example-0.1, 'NNNNNNNNNN', ha='left', va='center', fontsize=9,
         fontfamily='monospace', color='gray', fontweight='bold')
ax1.text(10.7, y_example-0.1, 'TATAAA', ha='left', va='center', fontsize=9,
         fontfamily='monospace', color='red', fontweight='bold')

ax1.text(6.5, y_example-0.7, 'Opposite (5 bases):', ha='left', va='center', fontsize=10, fontfamily='monospace')
ax1.text(6.5, y_example-1.1, 'GCCAAT', ha='left', va='center', fontsize=9,
         fontfamily='monospace', color='blue', fontweight='bold')
ax1.text(8.3, y_example-1.1, 'NNNNN', ha='left', va='center', fontsize=9,
         fontfamily='monospace', color='gray', fontweight='bold')
ax1.text(9.5, y_example-1.1, 'TATAAA', ha='left', va='center', fontsize=9,
         fontfamily='monospace', color='red', fontweight='bold')

# ===== Figure 2: Promoter Library Generation Flow =====
ax2 = axes[1]
ax2.set_xlim(0, 15)
ax2.set_ylim(0, 8)
ax2.axis('off')
ax2.set_title('Promoter Library Generation Workflow', fontsize=16, fontweight='bold', pad=20)

# Step 1: Input
y_step1 = 6.5
input1_box = FancyBboxPatch((0.5, y_step1), 2.5, 1, boxstyle="round,pad=0.1",
                             edgecolor='blue', facecolor='lightblue', linewidth=2)
ax2.add_patch(input1_box)
ax2.text(1.75, y_step1+0.7, 'Core CSV', ha='center', va='center', fontsize=11, fontweight='bold')
ax2.text(1.75, y_step1+0.3, 'e.g., 100 types', ha='center', va='center', fontsize=9)

input2_box = FancyBboxPatch((3.5, y_step1), 2.5, 1, boxstyle="round,pad=0.1",
                             edgecolor='red', facecolor='lightcoral', linewidth=2)
ax2.add_patch(input2_box)
ax2.text(4.75, y_step1+0.7, 'TFB CSV', ha='center', va='center', fontsize=11, fontweight='bold')
ax2.text(4.75, y_step1+0.3, 'e.g., 50 types', ha='center', va='center', fontsize=9)

# Step 2: Combination generation
arrow_down1 = FancyArrowPatch((3, y_step1-0.1), (3, y_step1-0.9),
                              arrowstyle='->', mutation_scale=20, linewidth=2.5, color='green')
ax2.add_patch(arrow_down1)

y_step2 = 4.5
combo_box = FancyBboxPatch((1, y_step2), 4, 1, boxstyle="round,pad=0.1",
                            edgecolor='purple', facecolor='plum', linewidth=2)
ax2.add_patch(combo_box)
ax2.text(3, y_step2+0.7, 'Generate All Combinations', ha='center', va='center', fontsize=12, fontweight='bold')
ax2.text(3, y_step2+0.3, '100 x 50 = 5,000 pairs', ha='center', va='center', fontsize=10)

# Step 3: Promoter design
arrow_down2 = FancyArrowPatch((3, y_step2-0.1), (3, y_step2-0.9),
                              arrowstyle='->', mutation_scale=20, linewidth=2.5, color='green')
ax2.add_patch(arrow_down2)

y_step3 = 2.5
design_box = FancyBboxPatch((0.5, y_step3), 5, 1, boxstyle="round,pad=0.1",
                             edgecolor='orange', facecolor='moccasin', linewidth=2)
ax2.add_patch(design_box)
ax2.text(3, y_step3+0.7, 'Design Promoters for Each Pair', ha='center', va='center', fontsize=12, fontweight='bold')
ax2.text(3, y_step3+0.3, 'Coplanar + Opposite', ha='center', va='center', fontsize=10)

# Step 4: Output
arrow_down3 = FancyArrowPatch((3, y_step3-0.1), (3, y_step3-0.9),
                              arrowstyle='->', mutation_scale=20, linewidth=2.5, color='green')
ax2.add_patch(arrow_down3)

y_step4 = 0.5
output_box = FancyBboxPatch((0.5, y_step4), 5, 1, boxstyle="round,pad=0.1",
                             edgecolor='darkgreen', facecolor='lightgreen', linewidth=2)
ax2.add_patch(output_box)
ax2.text(3, y_step4+0.7, 'Promoter Library (DataFrame)', ha='center', va='center', fontsize=12, fontweight='bold')
ax2.text(3, y_step4+0.3, '5,000 rows -> Save as CSV', ha='center', va='center', fontsize=10)

# Sample output table on the right
table_x = 7
table_y = 5
ax2.text(table_x+2, table_y+1.5, 'Output Example', ha='center', va='center', fontsize=11, fontweight='bold')

# Table header
header_data = ['ID', 'Core', 'TFB', 'Coplanar', 'Opposite']
col_widths = [1.2, 0.8, 0.8, 1.5, 1.5]
x_pos = table_x
for i, (header, width) in enumerate(zip(header_data, col_widths)):
    rect = patches.Rectangle((x_pos, table_y+0.5), width, 0.4, 
                              linewidth=1, edgecolor='black', facecolor='lightgray')
    ax2.add_patch(rect)
    ax2.text(x_pos+width/2, table_y+0.7, header, ha='center', va='center', fontsize=8, fontweight='bold')
    x_pos += width

# Table data (3 rows)
sample_data = [
    ['P_001', 'TATAAA', 'CAAT', 'CAAT...TATAAA', 'CAAT...TATAAA'],
    ['P_002', 'TATAAA', 'GCCAAT', 'GCCAAT...TATAAA', 'GCCAAT...TATAAA'],
    ['...', '...', '...', '...', '...'],
]

for row_idx, row_data in enumerate(sample_data):
    x_pos = table_x
    for col_idx, (cell, width) in enumerate(zip(row_data, col_widths)):
        rect = patches.Rectangle((x_pos, table_y+0.5-0.4*(row_idx+1)), width, 0.4,
                                  linewidth=1, edgecolor='black', facecolor='white')
        ax2.add_patch(rect)
        ax2.text(x_pos+width/2, table_y+0.7-0.4*(row_idx+1), cell, 
                 ha='center', va='center', fontsize=7, fontfamily='monospace')
        x_pos += width

plt.tight_layout()
plt.savefig('promoter_design_explanation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved as: promoter_design_explanation.png")

---

# プロモーターライブラリ生成コード

In [ ]:
# プロモーターライブラリ生成対応版

import pandas as pd
from itertools import product

def Coplanar(core_promoter, TFB, spacer_length=10):
    if not core_promoter or not TFB:
        raise ValueError("core_promoterとTFBは空であってはいけません")
    
    spacer = 'N' * spacer_length
    artificial_promoter = TFB + spacer + core_promoter
    
    return artificial_promoter


def Opposite(core_promoter, TFB, spacer_length=5):
    if not core_promoter or not TFB:
        raise ValueError("core_promoterとTFBは空であってはいけません")
    
    spacer = 'N' * spacer_length
    artificial_promoter = TFB + spacer + core_promoter
    
    return artificial_promoter


def design_promoter_with_custom_spacer(core_promoter, TFB, spacer_sequence):
    return TFB + spacer_sequence + core_promoter


def validate_sequence(sequence):
    valid_bases = set('ATGCNatgcn')
    return all(base in valid_bases for base in sequence)


def generate_promoter_library(core_csv=None, tfb_csv=None, 
                               core_list=None, tfb_list=None,
                               promoter_types=['Coplanar', 'Opposite'],
                               coplanar_spacer=10, opposite_spacer=5,
                               output_csv=None):
    """
    プロモーターライブラリを生成する関数
    
    CSVファイルまたはリストからcore_promoterとTFBを読み込み、
    全ての組み合わせで人工プロモーターを生成します。
    """
    
    # core_promoterの読み込み
    if core_csv:
        core_df = pd.read_csv(core_csv)
        # 'sequence'カラムがあると仮定、なければ最初のカラムを使用
        core_col = 'sequence' if 'sequence' in core_df.columns else core_df.columns[0]
        cores = core_df[core_col].tolist()
    elif core_list:
        cores = core_list
    else:
        raise ValueError("core_csvまたはcore_listを指定してください")
    
    # TFBの読み込み
    if tfb_csv:
        tfb_df = pd.read_csv(tfb_csv)
        tfb_col = 'sequence' if 'sequence' in tfb_df.columns else tfb_df.columns[0]
        tfbs = tfb_df[tfb_col].tolist()
    elif tfb_list:
        tfbs = tfb_list
    else:
        raise ValueError("tfb_csvまたはtfb_listを指定してください")
    
    # 全ての組み合わせを生成
    library = []
    
    for i, (core, tfb) in enumerate(product(cores, tfbs), 1):
        entry = {
            'ID': f'Promoter_{i:05d}',
            'Core_Promoter': core,
            'TFB': tfb,
        }
        
        if 'Coplanar' in promoter_types:
            coplanar_seq = Coplanar(core, tfb, spacer_length=coplanar_spacer)
            entry['Coplanar_Sequence'] = coplanar_seq
            entry['Coplanar_Length'] = len(coplanar_seq)
        
        if 'Opposite' in promoter_types:
            opposite_seq = Opposite(core, tfb, spacer_length=opposite_spacer)
            entry['Opposite_Sequence'] = opposite_seq
            entry['Opposite_Length'] = len(opposite_seq)
        
        library.append(entry)
    
    # DataFrameに変換
    library_df = pd.DataFrame(library)
    
    # CSVに保存（オプション）
    if output_csv:
        library_df.to_csv(output_csv, index=False)
        print(f"ライブラリを {output_csv} に保存しました")
    
    return library_df

---

# 使用例

In [ ]:
# サンプルデータでライブラリ生成のテスト

print("=" * 60)
print("人工プロモーターライブラリ生成ツール")
print("=" * 60)

# サンプルデータでテスト
print("\n【サンプルデータでライブラリ生成】")

sample_cores = ["TATAAA", "TATAAT", "TTGACA", "TATAAATA"]
sample_tfbs = ["CAAT", "GCCAAT", "CCAAT"]

library = generate_promoter_library(
    core_list=sample_cores,
    tfb_list=sample_tfbs,
    promoter_types=['Coplanar', 'Opposite'],
    coplanar_spacer=10,
    opposite_spacer=5,
    output_csv='promoter_library.csv'  # CSVファイルに保存
)

print(f"\n生成されたプロモーター数: {len(library)}")
print(f"組み合わせ: {len(sample_cores)} cores × {len(sample_tfbs)} TFBs = {len(library)} promoters")
print("\n最初の5件:")
print(library.head())

print("\n統計情報:")
if 'Coplanar_Length' in library.columns:
    print(f"Coplanar型の長さ範囲: {library['Coplanar_Length'].min()} - {library['Coplanar_Length'].max()} bp")
if 'Opposite_Length' in library.columns:
    print(f"Opposite型の長さ範囲: {library['Opposite_Length'].min()} - {library['Opposite_Length'].max()} bp")

print("\n" + "=" * 60)